# Seguridad del Data Lake

Endurecemos el acceso al bucket `datalake` del proyecto (zonas `raw/`, `curated/`,
`models/v1/`), que hoy se lee y escribe con credenciales de administrador de MinIO sin
ningÃºn control adicional.

**Riesgo:** **exfiltraciÃ³n** del modelo o de los datos crudos (cualquiera con las
credenciales de administrador puede bajar todo el bucket), y **manipulaciÃ³n del
artefacto**: si alguien reemplaza `models/v1/model.joblib` por una versiÃ³n alterada, el
servicio que lo carga (`tp_mlops2.predict.load_model_lightweight`, usado por Airflow) lo
servirÃ­a sin darse cuenta.

**Controles que agregamos:** checksum de integridad sobre el modelo real del proyecto,
URLs prefirmadas con expiraciÃ³n en vez de acceso directo con las credenciales maestras, y
una polÃ­tica de acceso mÃ­nimo diferenciada por zona.

In [1]:
import hashlib
import os
import tempfile

import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

BUCKET = "datalake"
ENDPOINT = os.environ.get("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(signature_version="s3v4"),
)

objetos = s3.list_objects_v2(Bucket=BUCKET, Prefix="models/v1/")["Contents"]
for o in objetos:
    print(o["Key"], f"{o['Size']:,} bytes")

models/v1/feature_cols.joblib 202 bytes
models/v1/model.joblib 32,625,425 bytes


## 1. Estado actual: sin checksum, acceso directo con credenciales maestras

El modelo real del proyecto (`models/v1/model.joblib`, ~32 MB) se sube y baja hoy sin
ningÃºn mecanismo que detecte si el archivo fue alterado entre una subida y la siguiente
lectura.

In [2]:
with tempfile.TemporaryDirectory() as tmp:
    ruta = f"{tmp}/model.joblib"
    s3.download_file(BUCKET, "models/v1/model.joblib", ruta)
    with open(ruta, "rb") as f:
        contenido = f.read()

print(f"Descargado sin verificar nada: {len(contenido):,} bytes")
print("Si este archivo fue reemplazado por una versiÃ³n manipulada, no hay forma de saberlo.")

Descargado sin verificar nada: 32,625,425 bytes
Si este archivo fue reemplazado por una versiÃ³n manipulada, no hay forma de saberlo.


## 2. Checksum de integridad sobre el modelo real

Calculamos el SHA-256 del modelo real del proyecto, lo publicamos junto al artefacto (tal
como ya existe `models/v1/feature_cols.joblib` al lado de `model.joblib`), y verificamos al
cargar.

In [3]:
def calcular_sha256(bucket: str, key: str) -> str:
    obj = s3.get_object(Bucket=bucket, Key=key)
    return hashlib.sha256(obj["Body"].read()).hexdigest()


sha_real = calcular_sha256(BUCKET, "models/v1/model.joblib")
print("SHA-256 real del modelo en producciÃ³n:", sha_real)

# lo publicamos junto al artefacto, mismo patrÃ³n que feature_cols.joblib
s3.put_object(Bucket=BUCKET, Key="models/v1/model.sha256", Body=sha_real.encode())
print("Publicado en models/v1/model.sha256")


def cargar_verificando(bucket: str, key_modelo: str, key_checksum: str) -> bytes:
    esperado = s3.get_object(Bucket=bucket, Key=key_checksum)["Body"].read().decode()
    blob = s3.get_object(Bucket=bucket, Key=key_modelo)["Body"].read()
    real = hashlib.sha256(blob).hexdigest()
    if real != esperado:
        raise ValueError(
            f"INTEGRIDAD ROTA: esperado {esperado[:16]}..., obtenido {real[:16]}... "
            "- el artefacto fue manipulado, no se carga."
        )
    return blob


# carga legÃ­tima: pasa
blob_ok = cargar_verificando(BUCKET, "models/v1/model.joblib", "models/v1/model.sha256")
print(f"\nCarga legÃ­tima verificada: {len(blob_ok):,} bytes, checksum coincide")

SHA-256 real del modelo en producciÃ³n:

 92dcd358b69d2fb4c2356126edca75da94a28daf14b1536d3b42ef506dcd8cd9
Publicado en models/v1/model.sha256

Carga legÃ­tima verificada: 32,625,425 bytes, checksum coincide


In [4]:
# simulamos manipulaciÃ³n: alguien sube una versiÃ³n alterada sin actualizar el checksum
s3.put_object(Bucket=BUCKET, Key="models/v1/model.joblib", Body=b"modelo-manipulado-por-un-atacante")

try:
    cargar_verificando(BUCKET, "models/v1/model.joblib", "models/v1/model.sha256")
except ValueError as e:
    print("detectado:", e)

# restauramos el modelo real para no dejar el bucket roto
s3.put_object(Bucket=BUCKET, Key="models/v1/model.joblib", Body=blob_ok)
print("\nmodelo real restaurado en el bucket")

detectado: INTEGRIDAD ROTA: esperado 92dcd358b69d2fb4..., obtenido 4e4da7851f3c308f... - el artefacto fue manipulado, no se carga.



modelo real restaurado en el bucket


## 3. URLs prefirmadas en vez de acceso directo con credenciales maestras

En vez de que cada servicio que necesite leer el modelo tenga las credenciales completas
de administrador de MinIO, se le entrega una URL temporal y acotada a un Ãºnico objeto.

In [5]:
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": "models/v1/model.joblib"},
    ExpiresIn=60,  # vÃ¡lida 60 segundos
)
print("URL prefirmada (expira en 60 s):")
print(url[:95], "...")

import requests

r = requests.get(url, timeout=10)
print(f"\nDescarga vÃ­a URL prefirmada: {r.status_code}, {len(r.content):,} bytes")

# polÃ­tica de acceso mÃ­nimo (documentada, no aplicada a nivel de IAM real en MinIO local):
# un rol de "serving" solo deberÃ­a poder LEER la zona de modelos, nunca escribir en raw/.
politica_acceso_minimo = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "SoloLecturaModelos",
            "Effect": "Allow",
            "Action": ["s3:GetObject"],
            "Resource": f"arn:aws:s3:::{BUCKET}/models/*",
        }
    ],
}
print("\nPolÃ­tica de acceso mÃ­nimo para el rol de serving:")
print(politica_acceso_minimo)

URL prefirmada (expira en 60 s):
http://localhost:9000/datalake/models/v1/model.joblib?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Cr ...

Descarga vÃ­a URL prefirmada: 200, 32,625,425 bytes

PolÃ­tica de acceso mÃ­nimo para el rol de serving:
{'Version': '2012-10-17', 'Statement': [{'Sid': 'SoloLecturaModelos', 'Effect': 'Allow', 'Action': ['s3:GetObject'], 'Resource': 'arn:aws:s3:::datalake/models/*'}]}


## 4. Zonas con permisos distintos, y un registro de acceso (gobernanza)

Las tres zonas del Data Lake de este proyecto tienen roles reales distintos: `raw/` solo
deberÃ­a recibir escritura del proceso de ingesta (`scripts/upload_to_datalake.py`),
`curated/` se lee pero no se reescribe fuera de ese mismo proceso, y `models/` es la Ãºnica
zona que necesita leer el servicio de predicciÃ³n. Documentamos esa separaciÃ³n y agregamos
un registro de acceso simple como medida de gobernanza.

In [6]:
REGISTRO_ACCESO: list[dict] = []


def registrar_acceso(operacion: str, key: str) -> None:
    import datetime

    REGISTRO_ACCESO.append(
        {
            "ts": datetime.datetime.now(datetime.UTC).isoformat(),
            "operacion": operacion,
            "key": key,
        }
    )


def get_object_auditado(bucket: str, key: str):
    registrar_acceso("get", key)
    return s3.get_object(Bucket=bucket, Key=key)


def put_object_auditado(bucket: str, key: str, body: bytes):
    registrar_acceso("put", key)
    return s3.put_object(Bucket=bucket, Key=key, Body=body)


# tres zonas, permisos documentados por rol (aplicado aquÃ­ vÃ­a el wrapper de auditorÃ­a,
# una polÃ­tica IAM real de MinIO/S3 en producciÃ³n implementarÃ­a esto a nivel de servidor)
ZONAS = {
    "raw/": "escritura solo para el proceso de ingesta",
    "curated/": "lectura general, escritura solo para el proceso de ingesta",
    "models/": "lectura para el servicio de predicciÃ³n, escritura solo al promover una versiÃ³n",
}
for zona, politica in ZONAS.items():
    print(f"{zona:12s} -> {politica}")

_ = get_object_auditado(BUCKET, "models/v1/feature_cols.joblib")
_ = get_object_auditado(BUCKET, "curated/energy_weather_clean.parquet")

print(f"\nEventos de acceso registrados: {len(REGISTRO_ACCESO)}")
for e in REGISTRO_ACCESO:
    print(" ", e["ts"][11:19], e["operacion"], e["key"])

raw/         -> escritura solo para el proceso de ingesta
curated/     -> lectura general, escritura solo para el proceso de ingesta
models/      -> lectura para el servicio de predicciÃ³n, escritura solo al promover una versiÃ³n

Eventos de acceso registrados: 2
  21:53:24 get models/v1/feature_cols.joblib
  21:53:24 get curated/energy_weather_clean.parquet


## 5. ReflexiÃ³n

**Â¿Por quÃ© una URL prefirmada con expiraciÃ³n es mÃ¡s segura que un bucket pÃºblico (o que
repartir las credenciales maestras)?**
Una URL prefirmada estÃ¡ acotada en tres dimensiones a la vez: a un objeto especÃ­fico (no a
todo el bucket), a una operaciÃ³n especÃ­fica (lectura, en este caso), y a una ventana de
tiempo corta. Si esa URL se filtra o queda en un log, el daÃ±o posible es leer un archivo
puntual durante unos segundos â€” muy distinto a filtrar las credenciales maestras de
`AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY`, que le darÃ­an a quien las obtenga acceso total
y permanente a las tres zonas del lake.

**Â¿QuÃ© ataque detecta el checksum del modelo?**
ManipulaciÃ³n del artefacto (parte del riesgo de *supply chain* en SAIF): alguien con
acceso de escritura al bucket â€” sea un atacante externo o un proceso interno con un bug â€”
podrÃ­a reemplazar el modelo servido sin tocar el cÃ³digo de ninguna API. El checksum hace
que esa sustituciÃ³n sea detectable en el momento de la carga, en vez de pasar
silenciosamente a producciÃ³n.

**Â¿QuÃ© elemento de SAIF cubre este control?**
El checksum cubre *asegurar la cadena de suministro*: el modelo no es un artefacto que se
construye y usa en un solo paso, viaja de MLflow al Data Lake y de ahÃ­ a quien lo sirve
(Airflow, en este proyecto, vÃ­a `load_model_lightweight`) â€” cada punto de ese trayecto es
una oportunidad para que algo se corrompa o sea manipulado, y verificar integridad en el
punto de consumo es lo que cierra esa cadena.